In [1]:

# CÀI ĐẶT VÀ IMPORT THƯ VIỆN

!pip install -q kaggle pandas numpy imbalanced-learn xgboost lightgbm pyarrow

import os
import glob
import zipfile

import pandas as pd
import numpy as np

print("Đã cài đặt/import thư viện.")



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\Cuong\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Đã cài đặt/import thư viện.


In [2]:

# KIỂM TRA KAGGLE API VÀ TẢI DATASET

kaggle_path = r"C:\Users\Cuong\Downloads\kaggle.json"

if not os.path.exists(kaggle_path):
    raise FileNotFoundError(
        f"Không tìm thấy file kaggle.json tại:\n{kaggle_path}"
    )

print(f"Đã tìm thấy Kaggle API: {kaggle_path}")

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

import shutil

shutil.copy(
    kaggle_path,
    os.path.expanduser("~/.kaggle/kaggle.json")
)

print("Đã sao chép kaggle.json vào thư mục Kaggle.")

# Tải dataset
print("\nĐang tải dataset từ Kaggle...")

!kaggle datasets download -d dhoogla/distrinetcicids2017

# Giải nén
os.makedirs("cicids2017_data", exist_ok=True)

zip_files = glob.glob("*.zip")

for file in zip_files:
    with zipfile.ZipFile(file, "r") as zip_ref:
        zip_ref.extractall("cicids2017_data")

    print(f"Đã giải nén: {file}")

Đã tìm thấy Kaggle API: C:\Users\Cuong\Downloads\kaggle.json
Đã sao chép kaggle.json vào thư mục Kaggle.

Đang tải dataset từ Kaggle...



  0%|          | 0.00/218M [00:00<?, ?B/s]
  0%|          | 1.00M/218M [00:03<14:16, 266kB/s]
  1%|          | 2.00M/218M [00:05<09:49, 384kB/s]
  1%|▏         | 3.00M/218M [00:09<11:55, 315kB/s]
  2%|▏         | 4.00M/218M [00:15<14:38, 255kB/s]
  2%|▏         | 5.00M/218M [00:22<18:27, 202kB/s]
  3%|▎         | 6.00M/218M [00:27<18:33, 199kB/s]
  3%|▎         | 7.00M/218M [00:33<19:37, 188kB/s]
  4%|▎         | 8.00M/218M [00:39<20:03, 183kB/s]
  4%|▎         | 8.00M/218M [00:50<20:03, 183kB/s]
  4%|▍         | 9.00M/218M [00:56<31:21, 116kB/s]
  5%|▍         | 10.0M/218M [01:00<26:33, 137kB/s]
  5%|▌         | 11.0M/218M [01:05<23:08, 156kB/s]
  6%|▌         | 12.0M/218M [01:11<22:49, 158kB/s]
  6%|▌         | 13.0M/218M [01:19<23:50, 150kB/s]
  6%|▋         | 14.0M/218M [01:26<23:33, 151kB/s]
  7%|▋         | 15.0M/218M [01:33<23:26, 151kB/s]
  7%|▋         | 16.0M/218M [01:38<21:37, 163kB/s]
  8%|▊         | 17.0M/218M [01:44<20:40, 170kB/s]
  8%|▊         | 18.0M/218M [01:52<22:

Dataset URL: https://www.kaggle.com/datasets/dhoogla/distrinetcicids2017
License(s): CC-BY-NC-SA-4.0

Đã giải nén: distrinetcicids2017.zip


In [3]:

#  KHẢO SÁT CÁC FILE DỮ LIỆU

# tất cả các file dữ liệu trong thư mục giải nén
data_files = sorted(
    glob.glob('cicids2017_data/**/*.csv', recursive=True) +
    glob.glob('cicids2017_data/**/*.parquet', recursive=True)
)

print(f"Tổng số file tìm thấy: {len(data_files)}")
print(f"Danh sách các file: {data_files}\n")

if data_files:
    # Lặp qua từng file để khảo sát chi tiết
    for idx, file_path in enumerate(data_files, 1):
        print("=" * 80)
        print(f" BÁO CÁO KHẢO SÁT FILE {idx}/{len(data_files)}: {file_path.split(os.sep)[-1]}")
        print("=" * 80)

        # Đọc dữ liệu (Hỗ trợ cả Parquet và CSV)
        if file_path.endswith('.csv'):
            df = pd.read_csv(file_path, low_memory=False)
        else:
            df = pd.read_parquet(file_path)

        # 1. Kích thước DataFrame
        num_rows, num_cols = df.shape
        print(f"1. Kích thước: {num_rows:,} dòng | {num_cols} cột")

        # 2. Kiểm tra dữ liệu trùng lặp
        num_duplicates = df.duplicated().sum()
        dup_percentage = (num_duplicates / num_rows) * 100 if num_rows > 0 else 0
        print(f"2. Số dòng bị trùng lặp (Duplicates): {num_duplicates:,} ({dup_percentage:.2f}%)")

        # 3. Thống kê dữ liệu thiếu
        missing_count = df.isnull().sum().sum()
        print(f"3. Tổng số giá trị thiếu (Missing values): {missing_count:,}")

        # 4. Phân loại cột theo kiểu dữ liệu
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        categorical_cols = df.select_dtypes(exclude=[np.number]).columns
        print("4. Phân loại thuộc tính:")
        print(f"   - Số lượng cột kiểu số (Numeric): {len(numeric_cols)}")
        print(f"   - Số lượng cột kiểu chữ (Categorical): {len(categorical_cols)}")

        if len(categorical_cols) > 0:
            print(f"   - Tên các cột kiểu chữ: {list(categorical_cols)}")

        # 5. Thống kê cột Label và phân bố nhãn
        label_candidates = [col for col in df.columns if 'label' in col.lower()]

        if label_candidates:
            label_col = label_candidates[0]
            label_counts = df[label_col].value_counts()

            print(f"5. Thống kê cột nhãn ('{label_col}'):")
            print(f"   - Số lượng nhãn khác nhau: {len(label_counts)}")
            print("   - Phân bố chi tiết từng nhãn:")

            for lbl, count in label_counts.items():
                print(f"     + {lbl}: {count:,} dòng")
        else:
            print("5. Không tìm thấy cột chứa từ khóa 'label'!")

        print("\n" + "-" * 80 + "\n")
else:
    print("Không tìm thấy file dữ liệu nào. Vui lòng kiểm tra lại quá trình giải nén.")


Tổng số file tìm thấy: 5
Danh sách các file: ['cicids2017_data\\Benign-Monday.parquet', 'cicids2017_data\\Bruteforce-Tuesday.parquet', 'cicids2017_data\\DoS-Wednesday.parquet', 'cicids2017_data\\Infiltration-Webattacks-Thursday.parquet', 'cicids2017_data\\Portscan-DDos-Botnet-Friday.parquet']

 BÁO CÁO KHẢO SÁT FILE 1/5: Benign-Monday.parquet
1. Kích thước: 350,718 dòng | 83 cột
2. Số dòng bị trùng lặp (Duplicates): 0 (0.00%)
3. Tổng số giá trị thiếu (Missing values): 0
4. Phân loại thuộc tính:
   - Số lượng cột kiểu số (Numeric): 82
   - Số lượng cột kiểu chữ (Categorical): 1
   - Tên các cột kiểu chữ: ['Label']
5. Thống kê cột nhãn ('Label'):
   - Số lượng nhãn khác nhau: 1
   - Phân bố chi tiết từng nhãn:
     + Benign: 350,718 dòng

--------------------------------------------------------------------------------

 BÁO CÁO KHẢO SÁT FILE 2/5: Bruteforce-Tuesday.parquet
1. Kích thước: 307,071 dòng | 83 cột
2. Số dòng bị trùng lặp (Duplicates): 0 (0.00%)
3. Tổng số giá trị thiếu (Missi

In [4]:

#  GỘP 5 FILE PARQUET VÀ TIỀN XỬ LÝ

# Tìm tất cả các file parquet trong thư mục
data_files = sorted(
    glob.glob('cicids2017_data/**/*.parquet', recursive=True)
)

print(f"Đã tìm thấy {len(data_files)} file dữ liệu.")

df_list = []

# Đọc từng file, thêm cột 'Day' và đưa vào danh sách
for file_path in data_files:
    # Trích xuất tên ngày từ tên file
    file_name = os.path.basename(file_path)
    day_name = file_name.split('-')[1].split('.')[0]

    # Đọc file parquet
    df_temp = pd.read_parquet(file_path)

    # Thêm thuộc tính 'Day'
    df_temp['Day'] = day_name

    df_list.append(df_temp)
    print(f"Đã xử lý xong file: {file_name} | Số dòng: {df_temp.shape[0]:,}")

# Gộp 5 file lại với nhau thành một DataFrame tổng
df_merged = pd.concat(df_list, ignore_index=True)

print(f"\nTổng số dòng sau khi gộp 5 file: {df_merged.shape[0]:,}")
print(f"Tổng số cột sau khi gộp (bao gồm cột Day): {df_merged.shape[1]}")

# Xóa các dòng mang nhãn 'Attempted-relabel-as-Benign'
target_col = 'Label'
initial_rows = len(df_merged)

df_merged = df_merged[
    df_merged[target_col] != 'Attempted-relabel-as-Benign'
]

print(
    f"Đã xóa {initial_rows - len(df_merged):,} dòng chứa nhãn 'Attempted'. "
    f"Số dòng còn lại: {len(df_merged):,}"
)

# Đánh nhãn lại theo bài toán nhị phân
df_merged['Binary_Label'] = df_merged[target_col].apply(
    lambda x: 1 if x.strip().lower() == 'benign' else 0
)

# Gom nhóm các thuộc tính giống nhau và nhãn giống nhau
feature_cols = [
    col for col in df_merged.columns
    if col not in [target_col, 'Binary_Label', 'Day']
]

print("\nĐang tiến hành gom nhóm các dòng có cùng đặc trưng và cùng nhãn...")

df_grouped = (
    df_merged
    .groupby(feature_cols + ['Binary_Label', 'Day'])
    .size()
    .reset_index(name='Duplicate_Count')
)

print(f"Số lượng dòng sau khi gom nhóm các đặc trưng giống nhau: {len(df_grouped):,}")
print("\nBảng phân phối nhãn nhị phân sau khi xử lý:")
print(df_grouped['Binary_Label'].value_counts())

display(df_grouped.head())


Đã tìm thấy 5 file dữ liệu.
Đã xử lý xong file: Benign-Monday.parquet | Số dòng: 350,718
Đã xử lý xong file: Bruteforce-Tuesday.parquet | Số dòng: 307,071
Đã xử lý xong file: DoS-Wednesday.parquet | Số dòng: 477,871
Đã xử lý xong file: Infiltration-Webattacks-Thursday.parquet | Số dòng: 281,439
Đã xử lý xong file: Portscan-DDos-Botnet-Friday.parquet | Số dòng: 370,259

Tổng số dòng sau khi gộp 5 file: 1,787,358
Tổng số cột sau khi gộp (bao gồm cột Day): 84
Đã xóa 8,966 dòng chứa nhãn 'Attempted'. Số dòng còn lại: 1,778,392

Đang tiến hành gom nhóm các dòng có cùng đặc trưng và cùng nhãn...
Số lượng dòng sau khi gom nhóm các đặc trưng giống nhau: 1,778,392

Bảng phân phối nhãn nhị phân sau khi xử lý:
Binary_Label
1    1496505
0     281887
Name: count, dtype: int64


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Idle Mean,Idle Std,Idle Max,Idle Min,ICMP Code,ICMP Type,Total TCP Flow Time,Binary_Label,Day,Duplicate_Count
0,0,2,2,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,-1,-1,0,1,DDos,1
1,0,2,2,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,-1,-1,0,1,Monday,1
2,0,2,2,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,-1,-1,0,1,Tuesday,1
3,0,2,2,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,-1,-1,0,1,Webattacks,1
4,0,2,2,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,-1,-1,0,1,Wednesday,1


In [5]:

# KIỂM TRA TỔNG QUAN SAU KHI GỘP & XỬ LÝ

print("=" * 70)
print(" BÁO CÁO KIỂM TRA LẠI DỮ LIỆU SAU KHI TIỀN XỬ LÝ ")
print("=" * 70)

# Kích thước DataFrame
print(f"1. Tổng số dòng hiện tại: {df_merged.shape[0]:,}")
print(f"2. Tổng số cột hiện tại: {df_merged.shape[1]} (bao gồm đặc trưng, Day và Binary_Label)")

# Kiểm tra giá trị thiếu
total_missing = df_merged.isnull().sum().sum()
print(f"3. Tổng số giá trị thiếu (NaN) trong toàn bộ bảng: {total_missing:,}")

# Phân loại kiểu dữ liệu cột
numeric_cols = df_merged.select_dtypes(include=[np.number]).columns
categorical_cols = df_merged.select_dtypes(exclude=[np.number]).columns

print("4. Phân loại kiểu dữ liệu:")
print(f"   - Số lượng cột kiểu số (Numeric): {len(numeric_cols)}")
print(f"   - Số lượng cột kiểu chữ/khác (Categorical): {len(categorical_cols)}")

if len(categorical_cols) > 0:
    print(f"   - Tên các cột kiểu chữ: {list(categorical_cols)}")

# 4. Kiểm tra phân phối nhãn nhị phân
print("\n5. Thống kê phân phối nhãn nhị phân (Binary_Label):")

binary_counts = df_merged['Binary_Label'].value_counts()

for val, count in binary_counts.items():
    label_name = "Benign (Bình thường)" if val == 1 else "Attack (Tấn công)"
    percentage = (count / len(df_merged)) * 100
    print(
        f"   - Nhãn {val} [{label_name}]: "
        f"{count:,} dòng ({percentage:.2f}%)"
    )

# 5. Kiểm tra phân bố dữ liệu theo từng ngày
if 'Day' in df_merged.columns:
    print("\n6. Phân bố số lượng mẫu theo từng ngày (Cột 'Day'):")

    day_counts = df_merged['Day'].value_counts()

    for day, count in day_counts.items():
        print(f"   - {day}: {count:,} dòng")

print("=" * 70)


 BÁO CÁO KIỂM TRA LẠI DỮ LIỆU SAU KHI TIỀN XỬ LÝ 
1. Tổng số dòng hiện tại: 1,778,392
2. Tổng số cột hiện tại: 85 (bao gồm đặc trưng, Day và Binary_Label)
3. Tổng số giá trị thiếu (NaN) trong toàn bộ bảng: 0
4. Phân loại kiểu dữ liệu:
   - Số lượng cột kiểu số (Numeric): 83
   - Số lượng cột kiểu chữ/khác (Categorical): 2
   - Tên các cột kiểu chữ: ['Label', 'Day']

5. Thống kê phân phối nhãn nhị phân (Binary_Label):
   - Nhãn 1 [Benign (Bình thường)]: 1,496,505 dòng (84.15%)
   - Nhãn 0 [Attack (Tấn công)]: 281,887 dòng (15.85%)

6. Phân bố số lượng mẫu theo từng ngày (Cột 'Day'):
   - Wednesday: 472,139 dòng
   - DDos: 369,061 dòng
   - Monday: 350,718 dòng
   - Tuesday: 307,032 dòng
   - Webattacks: 279,442 dòng


In [ ]:

# HUẤN LUYỆN VÀ ĐÁNH GIÁ 3 MÔ HÌNH MACHINE LEARNING

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_recall_curve,
    auc,
    confusion_matrix
)

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb


# TIỀN XỬ LÝ DỮ LIỆU & MÃ HÓA CỘT 'Day'

print("Đang tiến hành mã hóa cột chữ 'Day' ...")

df_model = df_merged.copy()

df_encoded = pd.get_dummies(
    df_model,
    columns=['Day'],
    drop_first=True
)

cols_to_drop = ['Label', 'Binary_Label']

X = df_encoded.drop(columns=cols_to_drop)
y = df_encoded['Binary_Label']

X.columns = [str(col).strip() for col in X.columns]


# CHIA TẬP DỮ LIỆU 70% TRAIN / 15% VAL / 15% TEST

print("Đang chia tập dữ liệu theo tỷ lệ 70/15/15 ...")

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print(f"- Kích thước tập Train: {X_train.shape[0]:,} dòng")
print(f"- Kích thước tập Val:   {X_val.shape[0]:,} dòng")
print(f"- Kích thước tập Test:  {X_test.shape[0]:,} dòng")


# CẤU HÌNH PIPELINE VÀ 3 MÔ HÌNH ML
k_features = 40

over = SMOTE(
    sampling_strategy=0.4,
    random_state=42
)

under = RandomUnderSampler(
    sampling_strategy=0.7,
    random_state=42
)

models = {
    "1. Logistic Regression":
        LogisticRegression(max_iter=1000, random_state=42),

    "2. Random Forest":
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ),

    "3. XGBoost":
        xgb.XGBClassifier(
            eval_metric='logloss',
            random_state=42,
            n_jobs=-1
        )
}


Đang tiến hành mã hóa cột chữ 'Day' (One-Hot Encoding)...
Đang chia tập dữ liệu theo tỷ lệ 70/15/15 ...
- Kích thước tập Train: 1,244,874 dòng
- Kích thước tập Val:   266,759 dòng
- Kích thước tập Test:  266,759 dòng


In [7]:
# HUẤN LUYỆN VÀ ĐÁNH GIÁ

results = []

for name, model in models.items():
    print(f"\n--- Đang huấn luyện mô hình: {name} ---")

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=f_classif, k=k_features)),
        ('over', over),
        ('under', under),
        ('classifier', model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(
        y_test,
        y_pred,
        average='binary',
        pos_label=0
    )
    macro_f1 = f1_score(
        y_test,
        y_pred,
        average='macro'
    )
    recall_attack = recall_score(
        y_test,
        y_pred,
        pos_label=0
    )

    cm = confusion_matrix(y_test, y_pred)
    TN, FP, FN, TP = cm.ravel()

    fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
    fnr = FN / (FN + TP) if (FN + TP) > 0 else 0

    precision_vals, recall_vals, _ = precision_recall_curve(
        y_test,
        y_proba,
        pos_label=0
    )

    pr_auc = auc(
        recall_vals,
        precision_vals
    )

    results.append({
        'Model': name,
        'Recall (Attack)': recall_attack,
        'F1-Score': f1,
        'Macro-F1': macro_f1,
        'PR-AUC': pr_auc,
        'FPR': fpr,
        'FNR': fnr,
        'Accuracy (Phụ)': acc,
        'Confusion Matrix (TN, FP, FN, TP)': (TN, FP, FN, TP)
    })

    print(f"-> Hoàn tất huấn luyện và đánh giá cho {name}!")


--- Đang huấn luyện mô hình: 1. Baseline: Logistic Regression ---


C:\Users\Cuong\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [33] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\Cuong\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


-> Hoàn tất huấn luyện và đánh giá cho 1. Baseline: Logistic Regression!

--- Đang huấn luyện mô hình: 2. Tabular ML 1: Random Forest ---


C:\Users\Cuong\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [33] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\Cuong\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


-> Hoàn tất huấn luyện và đánh giá cho 2. Tabular ML 1: Random Forest!

--- Đang huấn luyện mô hình: 3. Tabular ML 2: XGBoost ---


C:\Users\Cuong\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [33] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\Cuong\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


-> Hoàn tất huấn luyện và đánh giá cho 3. Tabular ML 2: XGBoost!


In [ ]:
# TỔNG HỢP VÀ IN BÁO CÁO KẾT QUẢ

results_df = pd.DataFrame(results)

print("\n" + "=" * 95)
print(" BẢNG SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH TRÊN TẬP TEST ")
print("=" * 95)

cols_to_display = [
    'Model',
    'Recall (Attack)',
    'F1-Score',
    'Macro-F1',
    'PR-AUC',
    'FPR',
    'FNR',
    'Accuracy'
]

print(
    results_df[cols_to_display]
    .to_string(index=False)
)

print("=" * 95)

print("\nCHI TIẾT CONFUSION MATRIX (TN, FP, FN, TP):")

for idx, row in results_df.iterrows():
    print(
        f"- {row['Model']}: "
        f"{row['Confusion Matrix (TN, FP, FN, TP)']}"
    )


 BẢNG SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH TRÊN TẬP TEST 
                           Model  Recall (Attack)  F1-Score  Macro-F1   PR-AUC      FPR      FNR  Accuracy (Phụ)
1. Baseline: Logistic Regression         0.976894  0.966460  0.980030 0.083826 0.023106 0.008420        0.989252
  2. Tabular ML 1: Random Forest         0.999740  0.997923  0.998765 0.079638 0.000260 0.000735        0.999340
        3. Tabular ML 2: XGBoost         0.999693  0.997805  0.998695 0.083808 0.000307 0.000771        0.999303

CHI TIẾT CONFUSION MATRIX (TN, FP, FN, TP):
- 1. Baseline: Logistic Regression: (np.int64(41306), np.int64(977), np.int64(1890), np.int64(222586))
- 2. Tabular ML 1: Random Forest: (np.int64(42272), np.int64(11), np.int64(165), np.int64(224311))
- 3. Tabular ML 2: XGBoost: (np.int64(42270), np.int64(13), np.int64(173), np.int64(224303))


In [ ]:

# LƯU KẾT QUẢ HUẤN LUYỆN


import os
import joblib

# Tạo thư mục lưu kết quả
os.makedirs("results", exist_ok=True)
os.makedirs("results/models", exist_ok=True)


# Lưu bảng kết quả đánh giá

results_df.to_csv(
    "results/model_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu kết quả đánh giá:")
print("  results/model_results.csv")


# Lưu từng mô hình đã train

trained_models = {}

for name, model in models.items():

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=f_classif, k=k_features)),
        ('over', over),
        ('under', under),
        ('classifier', model)
    ])

    pipeline.fit(X_train, y_train)

    # Tên file hợp lệ
    file_name = (
        name
        .replace(" ", "_")
        .replace(":", "")
        .replace(".", "")
    )

    model_path = f"results/models/{file_name}.pkl"

    joblib.dump(
        pipeline,
        model_path
    )

    trained_models[name] = pipeline

    print(f"Đã lưu mô hình: {model_path}")



# Lưu danh sách tên feature

joblib.dump(
    list(X.columns),
    "results/models/feature_columns.pkl"
)

print("Đã lưu danh sách feature.")


# Lưu tập kết quả dự đoán trên Test

test_results = X_test.copy()

test_results["Actual"] = y_test.values

for name, pipeline in trained_models.items():

    file_name = (
        name
        .replace(" ", "_")
        .replace(":", "")
        .replace(".", "")
    )

    test_results[f"Pred_{file_name}"] = pipeline.predict(X_test)

test_results.to_csv(
    "results/test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu kết quả dự đoán:")
print("  results/test_predictions.csv")


print("\n" + "=" * 60)
print("HOÀN TẤT LƯU KẾT QUẢ")
print("=" * 60)

Đã lưu kết quả đánh giá:
  results/model_results.csv


C:\Users\Cuong\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [33] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\Cuong\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Đã lưu mô hình: results/models/1_Baseline_Logistic_Regression.pkl


C:\Users\Cuong\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [33] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\Cuong\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Đã lưu mô hình: results/models/2_Tabular_ML_1_Random_Forest.pkl


C:\Users\Cuong\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [33] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\Cuong\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Đã lưu mô hình: results/models/3_Tabular_ML_2_XGBoost.pkl
Đã lưu danh sách feature.
Đã lưu kết quả dự đoán:
  results/test_predictions.csv

HOÀN TẤT LƯU KẾT QUẢ
results/
├── model_results.csv
├── test_predictions.csv
└── models/
    ├── Baseline_Logistic_Regression.pkl
    ├── Tabular_ML_1_Random_Forest.pkl
    ├── Tabular_ML_2_XGBoost.pkl
    └── feature_columns.pkl
